# 33 · 检索评估：Recall / MRR / NDCG

> 调 chunk、top-k、索引之前，先学会**量化检索好不好**。否则一切“感觉好”都是幻觉。

**本文件覆盖知识点**：Recall / Precision / Hit Rate / Recall@K / Precision@K / MRR / NDCG / MAP

先准备一份“测试集 + 人工标注的相关文档”，这是评估的前提。

In [1]:
# 迷你评估集：query → 相关文档 id（人工标注）
test_set = [
    {'q':'什么是 RAG',        'rel':[0]},
    {'q':'怎么部署客服机器人', 'rel':[3]},
    {'q':'有哪些大模型',       'rel':[4]},
    {'q':'机器人怎么收费',     'rel':[5]},
]
corpus = ['RAG 检索增强生成', '向量数据库', '提示词工程', '支持公有云与私有化', 'qwen 系列', '分基础/专业/企业版']

## 先用大白话讲一遍（公式看不懂就先看这里）

一次检索只要数出三个数，指标基本就是小学算术：

- **该找的** = 人工标注为相关的文档（记作 G），比如 1 条；
- **找到的** = 系统返回的前 K 条（记作 R_K），比如 3 条；
- **对的** = 两者的交集，比如 1 条。

| 指标 | 大白话怎么算 | 举例：该找 1 条、返回 3 条、对了 1 条 |
|------|--------------|----------------------------------------|
| **Recall@K** | 对的 ÷ 该找的 | 1 ÷ 1 = **1.0** —— 该找的都找到了，没漏 |
| **Precision@K** | 对的 ÷ 返回条数 K | 1 ÷ 3 ≈ **0.33** —— 返回里只有 1/3 有用 |
| **Hit Rate@K** | 有没有至少对 1 条，有记 1、没记 0 | **1** |
| **RR** | 1 ÷ 第一个正确答案在第几位 | 排第 1 位 → 1/1 = **1.0**；排第 3 位 → 1/3 ≈ **0.33** |
| **NDCG@K** | 同样是命中，排第 1 位拿满权重，排第 3 位只拿一半权重，加起来再归一 | 见下方手算 |
| **MAP** | 每命中一条就记一次「此刻的对的 ÷ 看过的」，取平均再对各查询取平均 | 见下方手算 |

三个最容易搞混的点：

1. **Precision@K 的分母永远是 K**，不是系统实际返回的条数。返回不足 K 条就是「缺位」，照样算错，分数会被拉低。
2. **NDCG 的「折损」= 排名越靠后，这条命中越不值钱**。第 i 位的权重是 1/log2(i+1)：第 1 位 1.0、第 2 位 0.63、第 3 位 0.5、第 10 位 0.29。所以「两条相关文档，一条在第 1 位一条在第 3 位」和「都在第 1、2 位」得分不同。
3. **多查询时先各自算、再取算术平均** —— MRR、MAP 里的那个 M 就是 Mean（平均）。所以单条查询命中得再好，被其他查询的 0 分一平均也会掉下来，评估集一定要足够大。

> 下面代码块会把这几个数的中间过程**一条条打印出来**，对照着看就明白了；公式版在下一节。

## 精确计算公式

> **约定**：查询 $q$；检索返回**有序**列表 $R=(d_1,\dots,d_N)$（按得分降序，$d_1$ 最靠前），Top-K 即 $R_K=(d_1,\dots,d_K)$；$G$ 为该查询**人工标注的相关文档集合**（gold）；二元相关性 $\mathrm{rel}(d)=\mathbb{1}[d\in G]$；$\mathbb{1}[\cdot]$ 为指示函数（成立取 1，否则 0）。下列先按**单查询**计算，多查询取算术平均：$\text{Metric}=\frac{1}{|Q|}\sum_{q\in Q}\text{Metric}(q)$。

**1) Recall@K（召回率）：该找到的，找到了多少（别漏）**
$$\mathrm{Recall@K}=\frac{|R_K\cap G|}{|G|}=\frac{\sum_{i=1}^{K}\mathrm{rel}(d_i)}{|G|}$$

**2) Precision@K（精确率）：找到的，有多少是对的（别错）**
$$\mathrm{Precision@K}=\frac{|R_K\cap G|}{K}=\frac{\sum_{i=1}^{K}\mathrm{rel}(d_i)}{K}$$
分母恒为 $K$（**不是** $|R_K|$）：系统返回不足 $K$ 条时按缺位处理，该值会被拉低。

**3) Hit Rate@K（命中率）：有没有至少找对一条**
$$\mathrm{HitRate@K}=\mathbb{1}\bigl[\,|R_K\cap G|>0\,\bigr]=\max_{1\le i\le K}\mathrm{rel}(d_i)\ \in\{0,1\}$$

**4) MRR（Mean Reciprocal Rank）：第一个正确答案排多前**
单查询倒数排名（无命中记 0）：
$$\mathrm{RR}(q)=\frac{1}{\mathrm{rank}(q)},\qquad \mathrm{rank}(q)=\min\{\,i:\ d_i\in G\,\}$$
$$\mathrm{MRR}=\frac{1}{|Q|}\sum_{q\in Q}\mathrm{RR}(q)$$
若只看 Top-K，则当 $\mathrm{rank}(q)>K$ 时取 $\mathrm{RR}=0$（记 $\mathrm{MRR@K}$）。

**5) NDCG@K（归一化折损累计增益）：越相关、排得越前越好**
本课采用**二元增益**（相关记 1）：
$$\mathrm{DCG@K}=\sum_{i=1}^{K}\frac{\mathrm{rel}(d_i)}{\log_2(i+1)}$$
$$\mathrm{IDCG@K}=\sum_{i=1}^{\min(|G|,\,K)}\frac{1}{\log_2(i+1)}$$
$$\mathrm{NDCG@K}=\frac{\mathrm{DCG@K}}{\mathrm{IDCG@K}}\in[0,1]$$
若用**分级相关性** $g_i\in\{0,1,2,3\}$，把分子增益换成 $2^{g_i}-1$：$\mathrm{DCG@K}=\sum_{i=1}^{K}\frac{2^{g_i}-1}{\log_2(i+1)}$。
折扣项 $\log_2(i+1)$ 的 $i$ 从 **1** 开始（第 1 位不折损）；若代码里 $i$ 从 0 开始，必须写成 $\log_2(i+2)$。

**6) MAP（Mean Average Precision）：综合“排序质量”**
单查询平均精度（$G$ 为该查询的相关集）：
$$\mathrm{AP}(q)=\frac{1}{|G|}\sum_{i=1}^{N}\mathrm{rel}(d_i)\cdot\mathrm{Precision@}i=\frac{1}{|G|}\sum_{i:\,d_i\in G}\frac{\bigl|\{d_1,\dots,d_i\}\cap G\bigr|}{i}$$
$$\mathrm{MAP}=\frac{1}{|Q|}\sum_{q\in Q}\mathrm{AP}(q)$$
AP 奖励“相关文档排得靠前”；截断版记 $\mathrm{AP@K}$（只累加到第 $K$ 位，分母仍是 $|G|$；若改用 $\min(|G|,K)$ 作分母，须在报告里注明口径）。

> 公式与下方代码一一对应：`recall_at_k`→(1)、`precision_at_k`→(2)、`hit_rate`→(3)、`mrr`→(4)、`ndcg`→(5)、`map_at_k`→(6)。

In [2]:
import numpy as np

def fake_retrieve(q, k=3):  # 桩检索器，生产换成真实检索
    qs = set(q)
    return list(np.argsort(-np.array([len(qs & set(d)) for d in corpus]))[:k])

# ---------- 指标实现 ----------
def recall_at_k(retrieved, rel, k):          # 召回的∩相关 / 总相关
    return len(set(retrieved[:k]) & set(rel)) / max(len(rel), 1)

def precision_at_k(retrieved, rel, k):       # 召回的∩相关 / k
    return len(set(retrieved[:k]) & set(rel)) / k

def hit_rate(retrieved, rel, k):             # Top-k 是否至少命中 1 条
    return 1.0 if set(retrieved[:k]) & set(rel) else 0.0

def mrr(retrieved, rel):                     # 第一个命中的倒数名次
    for r, d in enumerate(retrieved, 1):
        if d in rel: return 1.0 / r
    return 0.0

def ndcg(retrieved, rel, k):                 # 归一化折损累计增益
    dcg = sum(1/np.log2(i+1) for i, d in enumerate(retrieved[:k], 1) if d in rel)
    idcg = sum(1/np.log2(i+1) for i in range(1, min(len(rel), k)+1))
    return dcg / idcg if idcg else 0.0

# 汇总指标
k = 3
agg = {'recall':[], 'mrr':[], 'ndcg':[], 'hit':[]}
for t in test_set:
    r = fake_retrieve(t['q'], k)
    agg['recall'].append(recall_at_k(r, t['rel'], k))
    agg['mrr'].append(mrr(r, t['rel']))
    agg['ndcg'].append(ndcg(r, t['rel'], k))
    agg['hit'].append(hit_rate(r, t['rel'], k))

for name, v in agg.items():
    print(f'{name:8s} = {np.mean(v):.3f}')

recall   = 0.250
mrr      = 0.250
ndcg     = 0.250
hit      = 0.250


In [3]:
# 公式(6) MAP / AP@K：上面列了知识点但没实现，这里补上（纯计算，不需要调用模型）
def average_precision(retrieved, rel, k=None):
    """AP(@K) = (1/|G|) * Σ_i rel(d_i) * Precision@i"""
    rel = set(rel)
    if not rel:
        return 0.0
    hits, total = 0, 0.0
    for i, d in enumerate(retrieved[:k] if k else retrieved, 1):
        if d in rel:
            hits += 1
            total += hits / i          # Precision@i：前 i 条里命中的比例
    return total / len(rel)            # 分母是 |G|，不是命中数

def map_at_k(test_set, retrieved_map, k=None):
    """MAP = 各查询 AP 的算术平均"""
    return float(np.mean([average_precision(retrieved_map[t['q']], t['rel'], k) for t in test_set]))

k = 3
retrieved_map = {t['q']: fake_retrieve(t['q'], k) for t in test_set}
print('AP@3 逐查询:', [round(average_precision(retrieved_map[t['q']], t['rel'], k), 3) for t in test_set])
print('MAP@3 = %.3f' % map_at_k(test_set, retrieved_map, k))
print('→ 与 Recall/NDCG 对照：Recall 看“别漏”，NDCG/MAP 看“排得好不好”。')

AP@3 逐查询: [1.0, 0.0, 0.0, 0.0]
MAP@3 = 0.250
→ 与 Recall/NDCG 对照：Recall 看“别漏”，NDCG/MAP 看“排得好不好”。


In [4]:
# 手算演示：把每个指标的中间量一条条打印出来（纯算术，不调用模型）
q = test_set[0]['q']
rel = set(test_set[0]['rel'])
retrieved = [int(d) for d in fake_retrieve(q, k)]  # 复用上面的桩检索器（k=3）
hit_pos = [i for i, d in enumerate(retrieved, 1) if d in rel]   # 第几位命中
n_hit = len(hit_pos)

print('查询：', q)
print('该找的(人工标注相关) ：', sorted(rel), '→ 共', len(rel), '条')
print('找到的(系统返回前 3 条)：', retrieved)
print('对的(取交集)         ： 命中', n_hit, '条，位于第', hit_pos or '—', '位')
print('-' * 46)
print('Recall@3    = 对的 %d ÷ 该找的 %d = %.3f' % (n_hit, len(rel), n_hit / len(rel)))
print('Precision@3 = 对的 %d ÷ K=3      = %.3f   ← 分母恒为 K，不是实际返回条数' % (n_hit, n_hit / k))
print('HitRate@3   = 命中 %d 条 → %s' % (n_hit, '1（至少中一条）' if n_hit else '0（一条没中）'))
print('RR          = 1 ÷ 第一个命中位置 → %.3f' % (1 / hit_pos[0] if hit_pos else 0.0))
print('              （前 3 条都没命中就记 0，这就是"第一个正确答案排多前"）')

print('')
print('NDCG：从第 1 位往下走，命中才加分，位置越靠后加得越少（权重 = 1/log2(i+1)）')
dcg = 0.0
for i, d in enumerate(retrieved, 1):
    w = 1 / np.log2(i + 1)
    gain = w if d in rel else 0.0
    dcg += gain
    print('  第 %d 位   权重 %.3f   %s   本次加分 %.3f' % (i, w, '命中 ✅' if d in rel else '未命中', gain))
idcg = sum(1 / np.log2(i + 1) for i in range(1, min(len(rel), k) + 1))
print('  DCG@3  = %.3f（实际排序的加权得分）' % dcg)
print('  IDCG@3 = %.3f（理想排序：相关文档全排最前时的得分）' % idcg)
print('  NDCG@3 = DCG ÷ IDCG = %.3f   ← 1.0 表示"排得和理想一样好"' % (dcg / idcg))

print('')
print('AP：从第 1 位往下走，每命中一条就记一次"此刻的对的 ÷ 看过的"，最后 ÷ 该找的条数')
hits, total = 0, 0.0
for i, d in enumerate(retrieved, 1):
    if d in rel:
        hits += 1
        total += hits / i
        print('  第 %d 位命中 → 此刻 Precision = %d/%d = %.3f，累计 %.3f' % (i, hits, i, hits / i, total))
print('  AP  = 累计 %.3f ÷ 该找的 %d 条 = %.3f' % (total, len(rel), total / len(rel)))
print('  MAP = 各查询 AP 的算术平均 = %.3f（4 条查询里只有第 1 条命中，所以被平均下来）'
      % map_at_k(test_set, retrieved_map, k))

查询： 什么是 RAG
该找的(人工标注相关) ： [0] → 共 1 条
找到的(系统返回前 3 条)： [0, 4, 2]
对的(取交集)         ： 命中 1 条，位于第 [1] 位
----------------------------------------------
Recall@3    = 对的 1 ÷ 该找的 1 = 1.000
Precision@3 = 对的 1 ÷ K=3      = 0.333   ← 分母恒为 K，不是实际返回条数
HitRate@3   = 命中 1 条 → 1（至少中一条）
RR          = 1 ÷ 第一个命中位置 → 1.000
              （前 3 条都没命中就记 0，这就是"第一个正确答案排多前"）

NDCG：从第 1 位往下走，命中才加分，位置越靠后加得越少（权重 = 1/log2(i+1)）
  第 1 位   权重 1.000   命中 ✅   本次加分 1.000
  第 2 位   权重 0.631   未命中   本次加分 0.000
  第 3 位   权重 0.500   未命中   本次加分 0.000
  DCG@3  = 1.000（实际排序的加权得分）
  IDCG@3 = 1.000（理想排序：相关文档全排最前时的得分）
  NDCG@3 = DCG ÷ IDCG = 1.000   ← 1.0 表示"排得和理想一样好"

AP：从第 1 位往下走，每命中一条就记一次"此刻的对的 ÷ 看过的"，最后 ÷ 该找的条数
  第 1 位命中 → 此刻 Precision = 1/1 = 1.000，累计 1.000
  AP  = 累计 1.000 ÷ 该找的 1 条 = 1.000
  MAP = 各查询 AP 的算术平均 = 0.250（4 条查询里只有第 1 条命中，所以被平均下来）


## 指标怎么读

| 指标 | 侧重 | 一句话 |
|------|------|--------|
| **Recall@K** | 召回率 | 该找到的找到了多少（别漏） |
| **Precision@K** | 精确率 | 找到的有多少是对的（别错） |
| **Hit Rate** | 命中率 | 有没有至少找对一条 |
| **MRR** | 首位质量 | 第一个正确答案排多前 |
| **NDCG** | 排序质量 | 越相关排越前，加权计分 |
| **MAP** | 综合 | 多查询的平均精度均值 |

> **怎么用**：同一份测试集上换 chunk_size / top-k / 是否混合检索 / 是否重排，比较 Recall@K 与 NDCG，用数据决策而不是猜。

## 小结

- 先造**评估集**（人工标注相关文档），指标才有意义；
- Recall 管“别漏”，NDCG/MRR 管“排得好”，Precision 管“别错”；
- 检索指标是所有上游优化的“验收尺”。